## 1. Preparation

Clone the course repository.  
Install gensim (package for using word2vec)

In [ ]:
!git clone https://github.com/SimoneRebora/CMCLS
!pip install gensim

Read the datasets

In [ ]:
import pandas as pd

first_df = pd.read_csv('/content/CMCLS/materials/HoD_annotated_dataset.csv')
first_df

In [ ]:
# Read the class curated dataset (60 passages)
class_df = pd.read_csv('/content/CMCLS/materials/Hands_on_5_curation_first_round.csv')
class_df

Merge them together

In [ ]:
# 1. Read the first curated dataset (100 passages)
df_1 = pd.read_csv(
    '/content/CMCLS/materials/HoD_annotated_dataset.csv',
    encoding='utf-8',
    encoding_errors='replace'
)[['Sentences', 'curation']]

# 2. Read the class curated dataset (60 passages) and align column name
df_2 = pd.read_csv(
    '/content/CMCLS/materials/Hands_on_5_curation_first_round.csv',
    encoding='utf-8',
    encoding_errors='replace'
)[['text', 'curation']].rename(columns={'text': 'Sentences'})

# 3. Merge into 160 passages
my_df = pd.concat([df_1, df_2], ignore_index=True)

# 4. Remove empty rows and standardize text/labels
my_df = my_df.dropna(subset=['Sentences', 'curation']).copy()
my_df['Sentences'] = my_df['Sentences'].astype(str).str.strip()
my_df['curation'] = my_df['curation'].astype(str).str.strip().str.upper()

# 5. Rename columns
df = my_df
df = df.rename(columns={
    'Sentences': 'text',
    'curation': 'label'
})

# Show
df

## 2. Load the embeddings

First, show all models available in Gensim

In [ ]:
import gensim.downloader as api

# print all model names
info = api.info()
print(list(info["models"]))

In [ ]:
# print details of one model
my_model_name = "word2vec-google-news-300"
print(info["models"][my_model_name])

In [ ]:
# print details of another model
my_model_name = "glove-wiki-gigaword-100"
print(info["models"][my_model_name])

Choose one model and load it

In [ ]:
# download a model (this will take a while)
my_model_name = "glove-wiki-gigaword-100"

# Load pre-trained vectors
kv = api.load(my_model_name)

Explore the model, by printing some embeddings

In [ ]:
# check the embedding of a word
my_word = "heart"

# print the vector
kv[my_word]

In [ ]:
# check the embedding of a sentence
my_sentence = "Heart of darkness"

# load necessary libraries
import re
import numpy as np

# Document vector = mean of word vectors
# Google News vectors are case-sensitive, so try the original word first,
# then fall back to lowercase.
def doc_vector(text):
    vecs = []
    for w in re.findall(r"\w+", text):
        if w in kv:
            vecs.append(kv[w])
        elif w.lower() in kv:
            vecs.append(kv[w.lower()])
    return np.mean(vecs, axis=0) if vecs else np.zeros(kv.vector_size)

# print vector
doc_vector(my_sentence)

## 3. Run the Machine Learning

First, prepare the dataset, by splitting into into train and test

In [ ]:
from sklearn.model_selection import train_test_split

# split dataset into train and test
y = (df["label"].str.upper() == "YES").astype(int).values
texts_train, texts_test, y_train, y_test = train_test_split(
    df["text"].tolist(), y, test_size=0.2, stratify=y, random_state=42
)

# convert the texts into embeddings
X_train = np.array([doc_vector(t) for t in texts_train])
X_test = np.array([doc_vector(t) for t in texts_test])

# Print the dimensions of our tables
print("Train table shape:", X_train.shape)
print("Test table shape:", X_test.shape)

In [ ]:
# check one embedding
my_ID = 11

print(texts_train[my_ID])
X_train[my_ID]

Train the model by using (again) logistic regression, and test it on the test set

In [ ]:
# Train and evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# train a logistic regression model
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)

# test it!
print(classification_report(y_test, clf.predict(X_test),
                            target_names=["NO", "YES"]))

Explore the classification errors with a confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score

y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
macro_f1 = f1_score(y_test, y_pred, average="macro")

fig, ax = plt.subplots(figsize=(8, 4))

disp = ConfusionMatrixDisplay(cm, display_labels=["NO", "YES"])
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

ax.set_title("Confusion Matrix")

# Text annotations on the right
annotation = (
    f"Model:  {my_model_name}\n"
    f"Macro F1-score:  {macro_f1:.2f}"
)

fig.text(
    0.68, 0.5, annotation,
    fontsize=11,
    verticalalignment="center",
    family="monospace"
)

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

Classify a new text

In [ ]:
my_sentence = "He was black and poor and ugly."

def predict(text):
    return "YES" if clf.predict(doc_vector(text).reshape(1, -1))[0] == 1 else "NO"

print(predict(my_sentence))

## Your turn

Try to repeat the operation with a different embedding model (among the ones available in Gensim). Does the performance improve?